In [1]:
# We'll learn how to:

# • Split a pandas object into pieces using one or more keys (in the form of func
# tions, arrays, or DataFrame column names)
# •
# • Calculate group summary statistics, like count, mean, or standard deviation, or a
# user-defined function
# •
# • Apply within-group transformations or other manipulations, like normalization,
# linear regression, rank, or subset selection
# •
# • Compute pivot tables and cross-tabulations
# •
# • Perform quantile analysis and other statistical group analyses

In [2]:
import numpy as np

In [3]:
import pandas as pd

##  How to Think About Group Operations

In [4]:
#split-apply-combine for describing group operations.
#for example lets say we have a dataframe grouped into rows and columns and once grouping is done 
#now we can go to next step and apply the desired function to each group after applying function
#we finally combine them all together 

In [5]:
df = pd.DataFrame({"key1": ["a","a",None, "b","a","a",None],
   ....:                    "key2" : pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
   ....:                    "data1" : np.random.standard_normal(7),
   ....:                    "data2" : np.random.standard_normal(7)})

In [6]:
df

,key1,key2,data1,data2
0,a,1,2.031212,-0.340636
1,a,2,0.052934,0.791605
2,NaN,1,0.494873,-0.610801
3,b,2,-1.025741,0.221986
4,a,1,1.147620,-1.989804
5,a,<NA>,-0.355838,0.065176
6,NaN,1,1.203510,-0.025132


In [7]:
#calculating mean od data1 using lables of key 1

In [8]:
grouped = df["data1"].groupby(df["key1"])
#this is method one accessing data1 and then calling groupby with key1

In [9]:
grouped

In [10]:
#this group variable is now groupby object i.e. now this object has all info that is required to perform some operations 

In [11]:
#getting mean 
grouped.mean()

key1
a    0.718982
b   -1.025741
Name: data1, dtype: float64

In [12]:
#now here first of all why name key1? because we use df["key1"] 
# we used the names from key1 and values from data1 to find the mean here 

In [13]:
#using multiple arrays as list 
means = df["data1"].groupby([df["key1"], df["key2"]]).mean()

In [14]:
means

key1  key2
a     1       1.589416
      2       0.052934
b     2      -1.025741
Name: data1, dtype: float64

In [15]:
# here grouped using two keys and result is hierarchial index consisting of unique pairs of keys

In [16]:
means.unstack()

key2,1,2
key1,,
a,1.589416,0.052934
b,NaN,-1.025741


In [17]:
states = np.array(["OH", "CA", "CA", "OH", "OH", "CA", "OH"])
#can be any arrays of same right length


In [18]:
years =  [2005, 2005, 2006, 2005, 2006, 2005, 2006]

In [19]:
L = df["data1"].groupby([states, years]).mean()

In [20]:
L

CA  2005   -0.151452
    2006    0.494873
OH  2005    0.502735
    2006    1.175565
Name: data1, dtype: float64

In [21]:
states

array(['OH', 'CA', 'CA', 'OH', 'OH', 'CA', 'OH'], dtype='<U2')

In [22]:
years

[2005, 2005, 2006, 2005, 2006, 2005, 2006]

In [23]:
#now here too the result is a hierarchical index of unique pairs of keys

In [24]:
L.unstack()

,2005,2006
CA,-0.151452,0.494873
OH,0.502735,1.175565


In [25]:
#upon unstacking L we got a dataframe here 
#note that you can also  pass columns names (all the python objects can be used )

In [26]:
df.groupby("key1").mean()

,key2,data1,data2
key1,,,
a,1.333333,0.718982,-0.368415
b,2.0,-1.025741,0.221986


In [27]:
df.groupby("key2", dropna = False).mean(numeric_only = True )

,data1,data2
key2,,
1,1.219304,-0.741593
2,-0.486404,0.506796
<NA>,-0.355838,0.065176


In [28]:
#so note here :
# if you use df.groupby("key2").mean() then you will get error saying that dtype is str but i don't know why that error was ther so 
#the fixed version is what u see here in the code above now there are two fixes i will remove and add them step by step for more easy understanding 

In [29]:
# df.groupby("key2").mean() this is giving error
df.groupby("key2").mean(numeric_only = True)
#the second one works because i used the only numeric 

,data1,data2
key2,,
1,1.219304,-0.741593
2,-0.486404,0.506796


In [30]:
#HERe key1 is called as nuisance column as it doesn't have numeric data and that is why its not present in the result above

In [31]:
#another groupby method is size, which returns a series containing group sizes 

In [32]:
df.groupby(["key1","key2"]).size()
#by default the missing values are not counted in the result to revert this use dropna = False

key1  key2
a     1       2
      2       1
b     2       1
dtype: int64

In [33]:
df.groupby(["key1","key2"], dropna = False).size()

key1  key2
a     1       2
      2       1
      <NA>    1
b     2       1
NaN   1       2
dtype: int64

In [34]:
df.groupby(["key1","key2"]).count()
#similar to size but counts non null values in each group 

data1  data2
key1 key2              
a    1         2      2
     2         1      1
b    2         1      1

** Iterating over Groups 

In [35]:
# groupby's object supports iteration (here iteration means going through  a collection of data one by one)

In [36]:
for name, group in df.groupby("key1"):
    print(name)
    print(group)

a
  key1  key2     data1     data2
0    a     1  2.031212 -0.340636
1    a     2  0.052934  0.791605
4    a     1  1.147620 -1.989804
5    a  <NA> -0.355838  0.065176
b
  key1  key2     data1     data2
3    b     2 -1.025741  0.221986


In [37]:
#in case of multiple keys first element in tuple = tuple key values:
for(k1,k2), group in df.groupby(["key1","key2"]):
    print((k1,k2))
    print(group)

('a', 1)
  key1  key2     data1     data2
0    a     1  2.031212 -0.340636
4    a     1  1.147620 -1.989804
('a', 2)
  key1  key2     data1     data2
1    a     2  0.052934  0.791605
('b', 2)
  key1  key2     data1     data2
3    b     2 -1.025741  0.221986


In [38]:
#computing dictionary of data pieces in one liner 
pieces = {name: group for name, group in df.groupby("key1")}

In [39]:
pieces

{'a':   key1  key2     data1     data2
 0    a     1  2.031212 -0.340636
 1    a     2  0.052934  0.791605
 4    a     1  1.147620 -1.989804
 5    a  <NA> -0.355838  0.065176,
 'b':   key1  key2     data1     data2
 3    b     2 -1.025741  0.221986}

In [40]:
pieces["b"]

,key1,key2,data1,data2
3,b,2,-1.025741,0.221986


In [41]:
result = pd.DataFrame({
    "key":  df[["key1", "key2"]].mean(numeric_only=True, axis=1),
    "data": df[["data1", "data2"]].mean(axis=1)
})

print(result)


    key      data
0   1.0  0.845288
1   2.0  0.422270
2   1.0 -0.057964
3   2.0 -0.401878
4   1.0 -0.421092
5  <NA> -0.145331
6   1.0  0.589189


**Selecting a Column or Subset of Columns**


In [42]:
df.groupby("key1")["data1"] # returns a series
df.groupby("key1")[["data2"]] #returns a dataframe

In [43]:
#or for performing the same operation we can 
df["data1"].groupby(df["key1"]) # returns a series
df[["data2"]].groupby(df["key1"]) #returns a dataframe

In [44]:
df.groupby(["key1", "key2"])[["data2"]].mean() 
#for aggregating only a few column 
#here the output is a dataframe

data2
key1 key2          
a    1    -1.165220
     2     0.791605
b    2     0.221986

In [45]:
#if a list or
#array is passed, or a grouped Series if only a single column name is passed as a scalar
s_grouped = df.groupby(["key1", "key2"])["data2"]

In [46]:
s_grouped

In [47]:
s_grouped.mean()

key1  key2
a     1      -1.165220
      2       0.791605
b     2       0.221986
Name: data2, dtype: float64

In [48]:
df

,key1,key2,data1,data2
0,a,1,2.031212,-0.340636
1,a,2,0.052934,0.791605
2,NaN,1,0.494873,-0.610801
3,b,2,-1.025741,0.221986
4,a,1,1.147620,-1.989804
5,a,<NA>,-0.355838,0.065176
6,NaN,1,1.203510,-0.025132


**Grouping with Dictionaries and Series

In [49]:
# Normally, groupby uses column values directly (like df.groupby("key1")).
# But pandas also lets you group using a mapping (dictionary or Series) that defines how labels should be grouped.

In [50]:
people= pd.DataFrame(np.random.standard_normal((5,5)),
                      columns=["a","b","c","d","e"],
                      index=["Joe", "Steve", "Wanda", "Jill", "Trey"])

In [51]:
people

,a,b,c,d,e
Joe,1.183318,0.524682,-0.525891,1.519781,0.332500
Steve,-0.434077,-0.564534,-0.282892,1.129012,-0.017044
Wanda,0.748836,1.059453,1.495788,-0.442752,1.268789
Jill,-0.995288,0.328664,1.268712,-0.184690,0.334527
Trey,1.179783,0.111438,2.754019,0.471192,0.820026


In [52]:
people.iloc[2:3,[1,3]] = np.nan #add a few NA values 

In [53]:
people

,a,b,c,d,e
Joe,1.183318,0.524682,-0.525891,1.519781,0.332500
Steve,-0.434077,-0.564534,-0.282892,1.129012,-0.017044
Wanda,0.748836,NaN,1.495788,NaN,1.268789
Jill,-0.995288,0.328664,1.268712,-0.184690,0.334527
Trey,1.179783,0.111438,2.754019,0.471192,0.820026


In [54]:
#sum the columns by group
mapping = {"a": "red", "b":"red", "c":"blue", "d":"blue", "e":"red", "f":"orange"}

In [55]:
# by_column = people.groupby(mapping, axis=1)  this was for old version of pandas so the fix for this version is :
by_column = people.T.groupby(mapping).sum().T
#here T refers to transpose ; why transpose? wouldn't it change the output? NO it won't and to avoid that we are transposing it again 
#and in linear algebra we are thought that (A**T)**T = A  that is why we used another transpose after sum so that we can get the original oreintation back 

In [56]:
by_column


,blue,red
Joe,0.993890,2.040500
Steve,0.846120,-1.015655
Wanda,1.495788,2.017625
Jill,1.084022,-0.332097
Trey,3.225211,2.111247


In [57]:
#now grouping with series

In [58]:
map_series = pd.Series(mapping)

In [59]:
map_series

a       red
b       red
c      blue
d      blue
e       red
f    orange
dtype: str

In [60]:
map_count = people.T.groupby(map_series).count().T

In [61]:
map_count

,blue,red
Joe,2,3
Steve,2,3
Wanda,1,2
Jill,2,3
Trey,2,3


In [62]:
map_sum = people.T.groupby(map_series).sum().T

In [63]:
map_sum

,blue,red
Joe,0.993890,2.040500
Steve,0.846120,-1.015655
Wanda,1.495788,2.017625
Jill,1.084022,-0.332097
Trey,3.225211,2.111247


In [64]:
#lesson learned : its better to use transpose instead of axis in sum , count or any other operation of groupby's object

#Grouping With Function

In [65]:
#any function with a group key will be called once per index value with return values being used as the group names

In [66]:
#we want the name lengthh from the previous used example 
people.groupby(len).sum()

,a,b,c,d,e
3,1.183318,0.524682,-0.525891,1.519781,0.332500
4,0.184495,0.440102,4.022731,0.286502,1.154553
5,0.314759,-0.564534,1.212896,1.129012,1.251745


In [67]:
#Mixing functions with arrays, dictionaries, or Series is not a problem, as everything
#gets converted to arrays internally

In [68]:
key_list = ["one", "one", "one","two", "two"]

In [69]:
people.groupby([len, key_list]).min()

,,a,b,c,d,e
3,one,1.183318,0.524682,-0.525891,1.519781,0.332500
4,two,-0.995288,0.111438,1.268712,-0.184690,0.334527
5,one,-0.434077,-0.564534,-0.282892,1.129012,-0.017044


**Grouping by Index Levels 

In [70]:
#ability to aggregate
#using one of the levels of an axis index

In [71]:
columns = pd.MultiIndex.from_arrays([["US", "US", "US", "JP", "JP"],
....:                                     
[1, 3, 5, 1, 3]],
....:                                     
names=["cty", "tenor"])

In [72]:
hier_df = pd.DataFrame(np.random.standard_normal((4,5)), columns =columns )

In [73]:
hier_df

cty          US                            JP          
tenor         1         3         5         1         3
0      0.591435  0.368614  0.320941  0.050573  1.197982
1     -0.833909  1.250110  1.039603  0.774733 -1.439924
2      0.753608  1.271097 -1.263803 -2.753643  1.408692
3     -1.236021  1.428867 -1.219909  0.912167  0.680651

In [74]:
columns


MultiIndex([('US', 1),
            ('US', 3),
            ('US', 5),
            ('JP', 1),
            ('JP', 3)],
           names=['cty', 'tenor'])

In [75]:
#To group by level, pass the level number or name using the level keyword

In [76]:
hier_df.T.groupby(level="cty").count().T #here the same changes as above don't use axis = "columns" just traspose 

cty,JP,US
0,2,3
1,2,3
2,2,3
3,2,3


** Data Aggregation 

In [77]:
df

,key1,key2,data1,data2
0,a,1,2.031212,-0.340636
1,a,2,0.052934,0.791605
2,NaN,1,0.494873,-0.610801
3,b,2,-1.025741,0.221986
4,a,1,1.147620,-1.989804
5,a,<NA>,-0.355838,0.065176
6,NaN,1,1.203510,-0.025132


In [78]:
grouped = df.groupby("key1")

In [79]:
grouped["data1"].nsmallest(2)
#the nsmallest Series method selects the smallest requested number of values from the data

key1   
a     5   -0.355838
      1    0.052934
b     3   -1.025741
Name: data1, dtype: float64

In [80]:
# To use your own aggregation functions, pass any function that aggregates an array to
# the aggregate method or its short alias agg

In [81]:
def peak_to_peak(arr):
    return arr.max() - arr.min()

In [82]:
grouped.agg(peak_to_peak)

,key2,data1,data2
key1,,,
a,1,2.38705,2.781409
b,0,0.00000,0.000000


In [83]:
grouped.describe()

key2                                             data1            ...  \
     count      mean      std  min  25%  50%  75%  max count      mean  ...   
key1                                                                    ...   
a      3.0  1.333333  0.57735  1.0  1.0  1.0  1.5  2.0   4.0  0.718982  ...   
b      1.0       2.0     <NA>  2.0  2.0  2.0  2.0  2.0   1.0 -1.025741  ...   

                         data2                                          \
           75%       max count      mean       std       min       25%   
key1                                                                     
a     1.368518  2.031212   4.0 -0.368415  1.178038 -1.989804 -0.752928   
b    -1.025741 -1.025741   1.0  0.221986       NaN  0.221986  0.221986   

                                    
           50%       75%       max  
key1                                
a    -0.137730  0.246783  0.791605  
b     0.221986  0.221986  0.221986  

[2 rows x 24 columns]

In [84]:
#describe works although they aren't aggregations

**Column-Wise and Multiple Function Application 

In [85]:
tips = pd.read_csv("tips.csv")

In [86]:
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [87]:
#adding tip percentage named tip_pct

In [88]:
tips["tip_pct"] = tips["tip"] / tips["total_bill"]

In [89]:
tips

,total_bill,tip,sex,smoker,day,time,size,tip_pct
0,16.99,1.01,Female,No,Sun,Dinner,2,0.059447
1,10.34,1.66,Male,No,Sun,Dinner,3,0.160542
2,21.01,3.50,Male,No,Sun,Dinner,3,0.166587
3,23.68,3.31,Male,No,Sun,Dinner,2,0.139780
4,24.59,3.61,Female,No,Sun,Dinner,4,0.146808
5,25.29,4.71,Male,No,Sun,Dinner,4,0.186240
6,8.77,2.00,Male,No,Sun,Dinner,2,0.228050
7,26.88,3.12,Male,No,Sun,Dinner,4,0.116071
8,15.04,1.96,Male,No,Sun,Dinner,2,0.130319
9,14.78,3.23,Male,No,Sun,Dinner,2,0.218539


In [90]:
tiped = tips.groupby(["sex","smoker"])

In [91]:
tiped

In [92]:
tiped_pct = tiped["tip_pct"]

In [93]:
tiped_pct.agg("mean")

sex     smoker
Female  No        0.103127
Male    No        0.168266
Name: tip_pct, dtype: float64

In [94]:
# If you pass a list of functions or function names instead, you get back a DataFrame
# with column names taken from the functions:

In [95]:
tiped_pct.agg(["mean","std", peak_to_peak])

,,mean,std,peak_to_peak
sex,smoker,,,
Female,No,0.103127,0.061773,0.087361
Male,No,0.168266,0.040466,0.111979


In [96]:
tiped_pct.agg([("average","mean"),("stdev", np.std)])

,,average,stdev
sex,smoker,,
Female,No,0.103127,0.043680
Male,No,0.168266,0.037853


In [97]:
# #With a DataFrame you have more options, as you can specify a list of functions
# to apply to all of the columns or different functions per column

In [98]:
functions = ["count", "min","max"]

In [99]:
result = tiped[["tip_pct", "total_bill"]].agg(functions)

In [100]:
result

tip_pct                     total_bill              
                count       min       max      count    min    max
sex    smoker                                                     
Female No           2  0.059447  0.146808          2  16.99  24.59
Male   No           8  0.116071  0.228050          8   8.77  26.88

In [101]:
#resulting DataFrame has hierarchical columns


In [102]:
result["tip_pct"]

,,count,min,max
sex,smoker,,,
Female,No,2,0.059447,0.146808
Male,No,8,0.116071,0.228050


In [103]:
result["total_bill"]

,,count,min,max
sex,smoker,,,
Female,No,2,16.99,24.59
Male,No,8,8.77,26.88


In [104]:
#list of tuples with cutsom names can be passed:
ftuples = [("average", "mean"), ("Variance", np.var)]

In [105]:
tiped[["tip_pct","total_bill"]].agg(ftuples)

tip_pct           total_bill           
                average  Variance    average   Variance
sex    smoker                                          
Female No      0.103127  0.001908   20.79000  14.440000
Male   No      0.168266  0.001433   18.22375  41.989873

In [106]:
# #to apply potentially different functions to one or more of
# the columns

In [107]:
tiped.agg({"tip": np.max, "size": "sum"})

,,tip,size
sex,smoker,,
Female,No,3.61,6
Male,No,4.71,22


In [108]:
tiped.agg({"tip_pct":["min","max","mean","std"],
           "size":"sum"})

tip_pct                               size
                    min       max      mean       std  sum
sex    smoker                                             
Female No      0.059447  0.146808  0.103127  0.061773    6
Male   No      0.116071  0.228050  0.168266  0.040466   22

In [109]:
tips

,total_bill,tip,sex,smoker,day,time,size,tip_pct
0,16.99,1.01,Female,No,Sun,Dinner,2,0.059447
1,10.34,1.66,Male,No,Sun,Dinner,3,0.160542
2,21.01,3.50,Male,No,Sun,Dinner,3,0.166587
3,23.68,3.31,Male,No,Sun,Dinner,2,0.139780
4,24.59,3.61,Female,No,Sun,Dinner,4,0.146808
5,25.29,4.71,Male,No,Sun,Dinner,4,0.186240
6,8.77,2.00,Male,No,Sun,Dinner,2,0.228050
7,26.88,3.12,Male,No,Sun,Dinner,4,0.116071
8,15.04,1.96,Male,No,Sun,Dinner,2,0.130319
9,14.78,3.23,Male,No,Sun,Dinner,2,0.218539


In [110]:
#dataframe will have hierarchial column only if multiple functions are applied to at least one column |

**Returning Aggregated Data Without Row Indexes

In [111]:
#to turn off the index that came from the aggregated data use as_index=False to groupby

In [112]:
# # tips.groupby(["size"], as_index = False).mean()
# tips.groupby("size", as_index=False).agg("mean")

In [113]:
numeric_cols = tips.select_dtypes(include="number").columns
#  filters the DataFrame to only numeric columns
# returns just the column names.

In [114]:
result = tips.groupby("size", as_index=False)[numeric_cols].mean()


In [115]:
result

,total_bill,tip,size,tip_pct
0,15.852000,2.302000,2.0,0.155227
1,15.675000,2.580000,3.0,0.163564
2,25.586667,3.813333,4.0,0.149706


## Apply: General split-apply-combine

In [116]:
#apply splits the object being manipulated into pieces, invokes the passed
#function on each piece, and then attempts to concatenate the pieces.

In [117]:
#aim: to select the top
#five tip_pct values by group

In [118]:
#. First, write a function that selects the rows with the largest values in a particular column

In [119]:
def top(df, n=5, column = "tip_pct"):
    return df.sort_values(column, ascending = False)[:n]
    

In [120]:
top(tips, n=6)

,total_bill,tip,sex,smoker,day,time,size,tip_pct
6,8.77,2.00,Male,No,Sun,Dinner,2,0.228050
9,14.78,3.23,Male,No,Sun,Dinner,2,0.218539
5,25.29,4.71,Male,No,Sun,Dinner,4,0.186240
2,21.01,3.50,Male,No,Sun,Dinner,3,0.166587
1,10.34,1.66,Male,No,Sun,Dinner,3,0.160542
4,24.59,3.61,Female,No,Sun,Dinner,4,0.146808


In [121]:
tips.groupby("tip").apply(top)

,,total_bill,sex,smoker,day,time,size,tip_pct
tip,,,,,,,,
1.01,0,16.99,Female,No,Sun,Dinner,2,0.059447
1.66,1,10.34,Male,No,Sun,Dinner,3,0.160542
1.96,8,15.04,Male,No,Sun,Dinner,2,0.130319
2.00,6,8.77,Male,No,Sun,Dinner,2,0.228050
3.12,7,26.88,Male,No,Sun,Dinner,4,0.116071
3.23,9,14.78,Male,No,Sun,Dinner,2,0.218539
3.31,3,23.68,Male,No,Sun,Dinner,2,0.139780
3.50,2,21.01,Male,No,Sun,Dinner,3,0.166587
3.61,4,24.59,Female,No,Sun,Dinner,4,0.146808


In [122]:
# first the tips dataframe is split into groups based on the vALUES of the tip 
# result therefore has a hierarchical index with an inner level that contains index values from the original DataFrame.

In [123]:
tips.groupby(["smoker", "sex"]).apply(top, n=1, column="total_bill")

total_bill   tip  day    time  size   tip_pct
smoker sex                                                    
No     Female 4       24.59  3.61  Sun  Dinner     4  0.146808
       Male   7       26.88  3.12  Sun  Dinner     4  0.116071

In [124]:
result = tips.groupby("sex")["tip_pct"].describe()

In [125]:
result

,count,mean,std,min,25%,50%,75%,max
sex,,,,,,,,
Female,2.0,0.103127,0.061773,0.059447,0.081287,0.103127,0.124967,0.146808
Male,8.0,0.168266,0.040466,0.116071,0.137415,0.163564,0.194314,0.228050


result.unstack("sex")

In [126]:
# #inside groupby describe() is just a shortcut for : def f(group):
#     return group.describe()
# grouped.apply(f)

** Suppressing the group keys 

In [127]:
#just use groups_keys= False to groupby for disabling the groupkey in hierarchical index 

In [128]:
tips.groupby("smoker", group_keys=False).apply(top)

,total_bill,tip,sex,day,time,size,tip_pct
6,8.77,2.00,Male,Sun,Dinner,2,0.228050
9,14.78,3.23,Male,Sun,Dinner,2,0.218539
5,25.29,4.71,Male,Sun,Dinner,4,0.186240
2,21.01,3.50,Male,Sun,Dinner,3,0.166587
1,10.34,1.66,Male,Sun,Dinner,3,0.160542


In [129]:
#here smoker group key doesn't appers because we have disabled it !!

**Quantile and Bucket Analysis

In [130]:
#combining pd.cut, pd.qcut with pd.groupby makes it convenient to perform bucket or quantile analysis on dataset 

In [131]:
frame = pd.DataFrame({"data1": np.random.standard_normal(1000),
                      "data2": np.random.standard_normal(1000)})

In [132]:
frame

,data1,data2
0,0.042806,-1.021546
1,0.442155,2.075621
2,0.412362,0.285448
3,0.547110,1.034022
4,-0.224892,-2.069586
...,...,...
995,-0.365916,0.160418
996,-0.158347,-2.009299
997,-0.532547,0.601464
998,-0.468057,-0.051406


In [133]:
frame.head()

,data1,data2
0,0.042806,-1.021546
1,0.442155,2.075621
2,0.412362,0.285448
3,0.547110,1.034022
4,-0.224892,-2.069586


In [134]:
frame.tail()

,data1,data2
995,-0.365916,0.160418
996,-0.158347,-2.009299
997,-0.532547,0.601464
998,-0.468057,-0.051406
999,0.027549,-1.080171


In [135]:
quartiles=pd.cut(frame["data1"], 4)

In [136]:
quartiles

0       (-0.156, 1.341]
1       (-0.156, 1.341]
2       (-0.156, 1.341]
3       (-0.156, 1.341]
4      (-1.653, -0.156]
             ...       
995    (-1.653, -0.156]
996    (-1.653, -0.156]
997    (-1.653, -0.156]
998    (-1.653, -0.156]
999     (-0.156, 1.341]
Name: data1, Length: 1000, dtype: category
Categories (4, interval[float64, right]): [(-3.156, -1.653] < (-1.653, -0.156] < (-0.156, 1.341] < (1.341, 2.838]]

In [137]:
quartiles.head(10)

0     (-0.156, 1.341]
1     (-0.156, 1.341]
2     (-0.156, 1.341]
3     (-0.156, 1.341]
4    (-1.653, -0.156]
5     (-0.156, 1.341]
6     (-0.156, 1.341]
7    (-1.653, -0.156]
8     (-0.156, 1.341]
9     (-0.156, 1.341]
Name: data1, dtype: category
Categories (4, interval[float64, right]): [(-3.156, -1.653] < (-1.653, -0.156] < (-0.156, 1.341] < (1.341, 2.838]]

In [138]:
#categorical object returned by cut can be passed directly to groupby

In [139]:
def get_stats(group):
   ....:     return pd.DataFrame(
   ....:         {"min": group.min(), "max": group.max(),
   ....:         "count": group.count(), "mean": group.mean()}
   ....:     )

In [140]:
grouped = frame.groupby(quartiles)

In [141]:
grouped.apply(get_stats)

min       max  count      mean
data1                                                      
(-3.156, -1.653] data1 -3.150392 -1.663888     42 -2.078004
                 data2 -1.662034  1.906741     42  0.093080
(-1.653, -0.156] data1 -1.645517 -0.158347    385 -0.783098
                 data2 -3.000795  3.093669    385  0.011574
(-0.156, 1.341]  data1 -0.153935  1.319974    489  0.522367
                 data2 -3.375405  2.653916    489 -0.016650
(1.341, 2.838]   data1  1.343562  2.837671     84  1.770240
                 data2 -1.872071  2.606117     84  0.122693

In [142]:
#same result can be obtained by
grouped.agg(["min", "max", "count", "mean"])

data1                               data2            \
                       min       max count      mean       min       max   
data1                                                                      
(-3.156, -1.653] -3.150392 -1.663888    42 -2.078004 -1.662034  1.906741   
(-1.653, -0.156] -1.645517 -0.158347   385 -0.783098 -3.000795  3.093669   
(-0.156, 1.341]  -0.153935  1.319974   489  0.522367 -3.375405  2.653916   
(1.341, 2.838]    1.343562  2.837671    84  1.770240 -1.872071  2.606117   

                                  
                 count      mean  
data1                             
(-3.156, -1.653]    42  0.093080  
(-1.653, -0.156]   385  0.011574  
(-0.156, 1.341]    489 -0.016650  
(1.341, 2.838]      84  0.122693

In [143]:
quartiles_samp = pd.qcut(frame["data1"], 4, labels = False)
#we using pd.qcut to get use the buckets of the equal size buckets based on the sample quartiles

In [144]:
quartiles_samp.head()

0    2
1    2
2    2
3    2
4    1
Name: data1, dtype: int64

##  Example: Filling Missing Values with Group-Specific Values

In [145]:
s = pd.Series(np.random.standard_normal(6))

In [146]:
s

0    0.730337
1    0.252971
2   -0.741644
3   -0.908651
4   -0.710885
5    1.638056
dtype: float64

In [147]:
s[::2] = np.nan

In [148]:
s

0         NaN
1    0.252971
2         NaN
3   -0.908651
4         NaN
5    1.638056
dtype: float64

In [149]:
#we will use fillna to put values on the place of NAN

In [150]:
s.fillna(s.mean()) #means fill NaN places in s with the mean of s 

0    0.327459
1    0.252971
2    0.327459
3   -0.908651
4    0.327459
5    1.638056
dtype: float64

In [151]:
#similarly we can use other operations like:
# s.fillna(s.max())
#s.fillna(s.min())
# s.fillna(s.sum())

In [152]:
#The fill value to vary by group.

In [153]:
#M1 to do is group data and use apply with a function that calls fillna on each data chunk.
#example below
states = ["Ohio", "New York", "Vermont", "Florida",
              "Oregon", "Nevada", "California", "Idaho"]

In [154]:
group_key = ["East", "East", "East", "East",
       "West", "West", "West", "West"]

In [155]:
data = pd.Series(np.random.standard_normal(8), index = states)

In [156]:
data

Ohio         -1.033268
New York     -1.618313
Vermont      -1.268646
Florida       0.810335
Oregon       -1.862385
Nevada       -0.930361
California   -1.719851
Idaho         0.599623
dtype: float64

In [157]:
 data[["Vermont", "Nevada", "Idaho",]]= np.nan

In [158]:
data

Ohio         -1.033268
New York     -1.618313
Vermont            NaN
Florida       0.810335
Oregon       -1.862385
Nevada             NaN
California   -1.719851
Idaho              NaN
dtype: float64

In [159]:
#grouping data
data.groupby(group_key).size()

East    4
West    4
dtype: int64

In [160]:
data.groupby(group_key).count()

East    3
West    2
dtype: int64

In [161]:
data.groupby(group_key).mean()

East   -0.613749
West   -1.791118
dtype: float64

In [162]:
#filling Na places with group means

In [163]:
def fill_mean(group):
    return group.fillna(group.mean())

In [164]:
data.groupby(group_key).apply(fill_mean)

East  Ohio         -1.033268
      New York     -1.618313
      Vermont      -0.613749
      Florida       0.810335
West  Oregon       -1.862385
      Nevada       -1.791118
      California   -1.719851
      Idaho        -1.791118
dtype: float64

In [165]:
#as groups have name attribute set we can predefine the fill values

In [166]:
fill_values =  {"East": 0.5, "West": -1}

In [167]:
def fill_func(group):
    return group.fillna(fill_values[group.name])

In [168]:
data.groupby(group_key).apply(fill_func)

East  Ohio         -1.033268
      New York     -1.618313
      Vermont       0.500000
      Florida       0.810335
West  Oregon       -1.862385
      Nevada       -1.000000
      California   -1.719851
      Idaho        -1.000000
dtype: float64

## Example: Random Sampling and Permutation

In [169]:
# WE will use sample method here 


In [170]:
suits = ["H", "S", "C", "D"]  # Hearts, Spades, Clubs, Diamonds

In [171]:
card_val = (list(range(1,11)) + [10] *3)*4#to ge the logic here use the deck of cards !

In [172]:
base_names = ["A"]+ list(range(2,11)) + ["J", "K","Q"]

In [173]:
cards=[]

In [174]:
for suit in suits:
    cards.extend(str(num)+ suit for num in base_names)

In [175]:
deck= pd.Series(card_val, index=cards)

In [176]:
deck.head(13)

AH      1
2H      2
3H      3
4H      4
5H      5
6H      6
7H      7
8H      8
9H      9
10H    10
JH     10
KH     10
QH     10
dtype: int64

In [177]:
#drawing a hand of 5 cards from the deck can be written as :
def draw(deck, n=5):
     return deck.sample(n)

In [178]:
draw(deck)

AD     1
7C     7
QC    10
KS    10
4S     4
dtype: int64

In [179]:
#we want two random cards from each suit 
def get_suit(card): #last letter is suit
    return card [-1]

In [180]:
deck.groupby(get_suit).apply(draw, n = 2)

C  7C      7
   AC      1
D  5D      5
   7D      7
H  10H    10
   2H      2
S  10S    10
   8S      8
dtype: int64

In [181]:
# #we could pass group_keys=False to drop the outer suit index, leaving
# in just the selected cards

## Example: Group Weighted Average and Correlation

In [182]:
df = pd.DataFrame({"category": ["a", "a", "a", "a","b", "b", "b", "b"],
                   "data": np.random.standard_normal(8),
                   "weights": np.random.uniform(size = 8)})

In [183]:
df

,category,data,weights
0,a,-0.737098,0.459625
1,a,0.364508,0.578605
2,a,0.671503,0.590136
3,a,1.573619,0.294903
4,b,-0.863462,0.496345
5,b,-0.261767,0.261109
6,b,0.460912,0.795984
7,b,-1.055495,0.589804


In [184]:
#operations between columns in a DataFrame or two Series, such as a group weighted average, are possible

In [185]:
grouped = df.groupby("category")

In [186]:
def get_wavg(group):
    return np.average(group["data"], weights = group["weights"])
    #“Take values from data, and use weights only to decide importance.”

In [187]:
grouped #groupby's object 

In [188]:
grouped.apply(get_wavg)

category
a    0.380841
b   -0.351141
dtype: float64

## Example :  Group-Wise Linear Regression

In [189]:
import statsmodels.api as sm

In [190]:
df = pd.DataFrame({
    "group": ["A","A","A","B","B","B"],
    "x": [1,2,3,1,2,3],
    "y": [2,4,6,3,5,7]
})


In [193]:
def regress(group):
    X = group["x"]
    y = group["y"]

    X = sm.add_constant(X)  # adds intercept

    model = sm.OLS(y, X).fit()

    return pd.Series({
        "intercept": model.params["const"],
        "slope": model.params["x"]
    })

In [194]:
result = df.groupby("group").apply(regress)

In [196]:
print(result)

          intercept  slope
group                     
A     -8.881784e-16    2.0
B      1.000000e+00    2.0


## Group Transforms and "Unwrapped" GroupBys

In [198]:
df = pd.DataFrame({'key':['a','b','c']*4, 'value' : np.arange(12.)})

In [199]:
df

,key,value
0,a,0.0
1,b,1.0
2,c,2.0
3,a,3.0
4,b,4.0
5,c,5.0
6,a,6.0
7,b,7.0
8,c,8.0
9,a,9.0


In [205]:
g = df.groupby('key')['value'] 
#Group by key, then select only the value column for operations

In [203]:
g.mean()

key
a    4.5
b    5.5
c    6.5
Name: value, dtype: float64

In [204]:
g

In [206]:
# we wanted to produce a Series of the same shape as df['value'] but
# with values replaced by the average grouped by 'key'. We can pass a function that
# computes the mean of a single group to transform

In [207]:
def get_mean(group):
    return group.mean()
    

In [209]:
g.transform(get_mean)

0     4.5
1     5.5
2     6.5
3     4.5
4     5.5
5     6.5
6     4.5
7     5.5
8     6.5
9     4.5
10    5.5
11    6.5
Name: value, dtype: float64

In [210]:
#for built in aggregation functions we can pass a string alias as with the groupby agg method

In [212]:
g.transform('mean')

0     4.5
1     5.5
2     6.5
3     4.5
4     5.5
5     6.5
6     4.5
7     5.5
8     6.5
9     4.5
10    5.5
11    6.5
Name: value, dtype: float64

In [213]:
#transform works with function to return series but the result must be the same size as the input 

In [214]:
#multiplying each group by helper function
def times_two(group):
    return group*2

In [215]:
g.transform(times_two)

0      0.0
1      2.0
2      4.0
3      6.0
4      8.0
5     10.0
6     12.0
7     14.0
8     16.0
9     18.0
10    20.0
11    22.0
Name: value, dtype: float64

In [216]:
#we can compute the ransk in descending order for each group

In [219]:
 def get_ranks(group):
   .....:     return group.rank(ascending=False)

In [220]:
g.transform(get_ranks)

0     4.0
1     4.0
2     4.0
3     3.0
4     3.0
5     3.0
6     2.0
7     2.0
8     2.0
9     1.0
10    1.0
11    1.0
Name: value, dtype: float64

In [221]:
def normalize(x):
    return (x-x.mean())/ x.std()

In [222]:
g.transform(normalize)

0    -1.161895
1    -1.161895
2    -1.161895
3    -0.387298
4    -0.387298
5    -0.387298
6     0.387298
7     0.387298
8     0.387298
9     1.161895
10    1.161895
11    1.161895
Name: value, dtype: float64

In [227]:
#built in aggregation functions are much faster than a general apply function, they also have fast path when used with transform 
#which allows us to perform unwrapped group operation

In [228]:
g.transform('mean')

0     4.5
1     5.5
2     6.5
3     4.5
4     5.5
5     6.5
6     4.5
7     5.5
8     6.5
9     4.5
10    5.5
11    6.5
Name: value, dtype: float64

In [229]:
normalized = (df['value']- g.transform('mean'))/ g.transform('std')

In [230]:
normalized

0    -1.161895
1    -1.161895
2    -1.161895
3    -0.387298
4    -0.387298
5    -0.387298
6     0.387298
7     0.387298
8     0.387298
9     1.161895
10    1.161895
11    1.161895
Name: value, dtype: float64

In [231]:
# Here, we are doing arithmetic between the outputs of multiple GroupBy operations
# instead of writing a function and passing it to groupby(...).apply. That is what is
# meant by “unwrapped.”

## Pivot Tables and Cross-Tabulation

In [232]:
# #A pivot table is a data summarization tool frequently found in spreadsheet programs
# and other data analysis software.

In [233]:
# #suppose you wanted to compute a table of group
# means (the default pivot_table aggregation type) arranged by day and smoker on
# the rows

In [234]:
tips.head()

,total_bill,tip,sex,smoker,day,time,size,tip_pct
0,16.99,1.01,Female,No,Sun,Dinner,2,0.059447
1,10.34,1.66,Male,No,Sun,Dinner,3,0.160542
2,21.01,3.50,Male,No,Sun,Dinner,3,0.166587
3,23.68,3.31,Male,No,Sun,Dinner,2,0.139780
4,24.59,3.61,Female,No,Sun,Dinner,4,0.146808


In [238]:
tips.pivot_table(
    values="tip",
    index=["day", "smoker"]
)

,,tip
day,smoker,
Sun,No,2.811


In [239]:
tips.pivot_table(index=["time", "day"], columns="smoker",
   .....:                  values=["tip_pct", "size"])

,,size,tip_pct
,smoker,No,No
time,day,,
Dinner,Sun,2.8,0.155238


In [242]:
tips.pivot_table(index=["time", "smoker"], columns="day",
   .....:                  values="tip_pct", aggfunc=len, margins=True)
#To use an aggregation function other than mean, pass it to the aggfunc keyword argument.

,day,Sun,All
time,smoker,,
Dinner,No,10,10
All,,10,10


** Cross-Tabulations: Crosstab

In [243]:
#special case of pivot table that computes group frequencies 

In [248]:
df = pd.DataFrame({
    "Class": ["10A", "10A", "10B", "10B", "10A", "10B"],
    "Result": ["Pass", "Fail", "Pass", "Pass", "Pass", "Fail"]
})

In [249]:

df

,Class,Result
0,10A,Pass
1,10A,Fail
2,10B,Pass
3,10B,Pass
4,10A,Pass
5,10B,Fail


In [250]:
pd.crosstab(df["Class"],df["Result"])

Result,Fail,Pass
Class,,
10A,1,2
10B,1,2


In [251]:
pd.crosstab(
    df["Class"],
    df["Result"],
    margins=True
)

Result,Fail,Pass,All
Class,,,
10A,1,2,3
10B,1,2,3
All,2,4,6


In [252]:
#margins = true  : adds grand total row and column !!

In [253]:
#THIS CHAPTER ENDS HERE NOW 